# Exp 13: Oversampled views

Card: [exp13_oversampled_views.md](../../experiments/exp13_oversampled_views.md)


In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from core.augment import RECIPES, Validation
from core.config import MEAN, ROOT, SEED, STD, TRAIN_DIR
from core.cv import VIP, Config
from core.data import KaggleDataModule

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_IMAGES = 1  # per class
    MAX_DRAWS = 2  # columns of draws
else:
    N_IMAGES = 8
    MAX_DRAWS = None  # all draws
CONFIG = Config.from_toml(
    ROOT / "sections" / "model" / "configs" / "exp11_resnet18_5fold" / "resnet18.toml"
)
FOLD = 0
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp13_oversampled_views"
SAVE_FIGURES = not SMOKE

plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.titlecolor": "#52514e",
    }
)

## One epoch of sampler


In [ ]:
torch.manual_seed(SEED)
dm = KaggleDataModule(
    CONFIG.read_data(),
    fold=FOLD,
    batch_size=CONFIG.batch_size,
    num_workers=0,
    train_tfms=RECIPES[CONFIG.augment](CONFIG.img_size).compose(),
    oversample=CONFIG.oversample,
)
dm.setup()
train = dm.train_ds.df
draws = Counter(dm.train_dataloader().sampler)  # dataset row: times drawn in epoch
vip = train[VIP].any(axis=1)
print(
    f"draws per image, mean: VIP {np.mean([draws[i] for i in train.index[vip]]):.2f}, "
    f"other {np.mean([draws[i] for i in train.index[~vip]]):.2f}"
)

## Views per VIP class


In [ ]:
def show(x):
    """Normalized CHW tensor back to HWC image in [0, 1]."""
    return (x.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)


resize = Validation(CONFIG.img_size).compose()
rng = np.random.default_rng(SEED)
for cls in VIP:
    rows = rng.choice(train.index[train[cls] == 1], N_IMAGES, replace=False)
    n_cols = 1 + (MAX_DRAWS or max(draws[i] for i in rows))
    fig, axes = plt.subplots(
        N_IMAGES, n_cols, figsize=(1.8 * n_cols, 2.0 * N_IMAGES), squeeze=False
    )
    for ax_row, i in zip(axes, rows):
        image = np.array(
            Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg").convert("RGB")
        )
        ax_row[0].imshow(show(resize(image=image)["image"]))
        ax_row[0].set_title(f"original · {draws[i]} draws", fontsize=8)
        for ax in ax_row[1 : 1 + min(draws[i], n_cols - 1)]:
            ax.imshow(
                show(dm.train_ds[i][0])
            )  # fresh augmentation per draw, as in training
        for ax in ax_row:
            ax.set_axis_off()
    fig.suptitle(
        f"{cls} · fold {FOLD} train · one epoch, oversample {CONFIG.oversample}, augment {CONFIG.augment!r}"
    )
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURES / f"{PREFIX}_{cls.lower()}.png", dpi=100)
    plt.show()